[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/21.%20The%20Yard%20Block%20Housekeeping%20Problem/P21-Tier-1_executed.ipynb)

---

# 21. The Yard Block Housekeeping Problem

## Tier 1: Network Flow Formulation

### Goal and scope

This notebook formulates yard housekeeping as a minimum-cost network-flow problem. It decides how many containers to move between yard blocks so that each block meets its stated net-flow requirement while its final load remains within capacity. An integer linear programming model expresses these requirements, and PuLP's CBC solver solves the model.

The notebook treats feasibility as an essential part of optimization. An optimum exists only if all modeled requirements can be satisfied simultaneously. The supplied demands and capacities are intentionally examined as given; if they conflict, the notebook reports the solver status rather than presenting an invalid movement plan.

### Inputs and outputs

Inputs are each block's current load, capacity, and signed demand, together with a directed movement-cost matrix. A negative demand means required net outflow; a positive demand means required net inflow. The model returns a solver status, a movement plan and its cost when an optimum exists, and final block loads. A sensitivity experiment also examines uniform multipliers applied to all movement costs.

### Modeling assumptions

- Containers are represented as integer TEU flows between blocks during one planning period.
- Each block has an exact signed net-flow requirement: positive demand is net inflow, and negative demand is net outflow.
- Each block's final load must be between zero and its capacity, inclusive.
- Movement costs are known, nonnegative, and constant during the planning period. The supplied matrix is symmetric, although the formulation permits directional costs.
- The model includes movement costs only. It excludes holding costs, time-dependent flows, equipment availability, handling limits, uncertainty, and movement sequencing.

### Example data

The example has four blocks:

| Block | Current load (TEU) | Capacity (TEU) | Demand (TEU) |
|---:|---:|---:|---:|
| 1 | 120 | 120 | -20 |
| 2 | 80 | 100 | +15 |
| 3 | 95 | 110 | +10 |
| 4 | 110 | 120 | +25 |

Inter-block movements transfer TEU between blocks, so the sum of all blocks' net outflows must be zero. The required net outflow is the negative of demand. Here, the demands sum to \(-20+15+10+25=30\) TEU, so the required net outflows sum to \(-30\) TEU, not zero. Thus the exact demands cannot all be met by movements among these blocks. Independently, Block 4's required net inflow of 25 TEU would produce a final load of \(110+25=135\) TEU, above its capacity of 120 TEU. These arithmetic checks establish that this instance is infeasible under the stated assumptions; they are not solver results.

### Solution sequence

1. Represent blocks and movement costs as data structures.
2. Create a nonnegative integer decision variable for every directed movement between distinct blocks.
3. Minimize total movement cost subject to exact flow-balance equations and final-load bounds.
4. Check the solver status before extracting or interpreting a plan.
5. Verify the plan when an optimum exists and examine uniform movement-cost multipliers.

This is a network-flow formulation, not a heuristic. Its linear constraints represent flow balance and capacity directly. Integer variables represent indivisible TEU movements. With integer data, the standard network-flow balance structure has integral extreme-point solutions; the explicit integer declaration also makes the intended decision domain clear. The solver status remains essential because no objective value can make mutually conflicting constraints feasible.

In [ ]:
# These libraries represent the input data, build the optimization model, and visualize its results.
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import pulp

plt.style.use("seaborn-v0_8")

In [ ]:
@dataclass
class Block:
    """Store a yard block's load, capacity, and signed net-flow requirement."""

    id: int
    current_load: int
    capacity: int
    demand: int

    @property
    def utilization(self) -> float:
        """Return the fraction of block capacity currently in use."""
        return self.current_load / self.capacity


@dataclass
class YardState:
    """Store block data and the directed movement-cost matrix."""

    blocks: List[Block]
    movement_costs: np.ndarray

    def __post_init__(self):
        self.num_blocks = len(self.blocks)
        expected_shape = (self.num_blocks, self.num_blocks)
        if self.movement_costs.shape != expected_shape:
            raise ValueError(
                f"Movement-cost matrix must have shape {expected_shape}."
            )
        block_ids = [block.id for block in self.blocks]
        if block_ids != list(range(1, self.num_blocks + 1)):
            raise ValueError(
                "Block IDs must be consecutive integers starting at 1, "
                "in the same order as the movement-cost matrix."
            )
        if not np.all(np.isfinite(self.movement_costs)):
            raise ValueError("Movement costs must be finite.")
        if np.any(self.movement_costs < 0):
            raise ValueError("Movement costs must be nonnegative.")
        for block in self.blocks:
            if block.capacity <= 0:
                raise ValueError("Block capacities must be positive.")
            if block.current_load < 0:
                raise ValueError("Current block loads cannot be negative.")

    def get_block_by_id(self, block_id: int) -> Block:
        """Return the block with the requested ID."""
        return next(block for block in self.blocks if block.id == block_id)

In [ ]:
def create_concrete_example() -> YardState:
    """Create the four-block example and its movement-cost matrix."""
    blocks = [
        Block(id=1, current_load=120, capacity=120, demand=-20),
        Block(id=2, current_load=80, capacity=100, demand=15),
        Block(id=3, current_load=95, capacity=110, demand=10),
        Block(id=4, current_load=110, capacity=120, demand=25),
    ]

    movement_costs = np.array(
        [
            [0, 15, 25, 30],
            [15, 0, 20, 35],
            [25, 20, 0, 18],
            [30, 35, 18, 0],
        ],
        dtype=float,
    )

    return YardState(blocks=blocks, movement_costs=movement_costs)


yard = create_concrete_example()
print("Yard configuration:")
for block in yard.blocks:
    print(
        f"Block {block.id}: {block.current_load}/{block.capacity} TEU "
        f"({block.utilization:.1%}), demand: {block.demand:+d} TEU"
    )

## Mathematical model

### Decision variables and intuition

A directed movement variable records how many TEU leave one block and arrive at another. At a block, outgoing flow minus incoming flow is its net outflow. Matching that quantity to the negative of signed demand makes a negative demand require net outflow and a positive demand require net inflow. Capacity bounds then check whether the resulting block load is physically allowed.

Let \(B=\{1,\ldots,n\}\) be the set of blocks. For each \(i\in B\), let \(L_i\) be its current load, \(U_i\) its capacity, and \(d_i\) its signed demand. Let \(c_{ij}\) be the cost per TEU moved from block \(i\) to block \(j\). For distinct \(i,j\in B\), let \(x_{ij}\) be the nonnegative integer number of TEU moved from \(i\) to \(j\). No self-movement variable is needed because moving a container within the same block changes neither its load nor the inter-block flow.

The objective is to minimize the sum of the cost of every directed movement:

$$
\min_{x}\quad \sum_{i\in B}\sum_{\substack{j\in B\\j\ne i}} c_{ij}x_{ij}.
$$

Each block must meet its exact signed net-flow requirement:

$$
\sum_{\substack{j\in B\\j\ne i}}x_{ij}
-\sum_{\substack{j\in B\\j\ne i}}x_{ji}
= -d_i,
\qquad i\in B.
$$

The first sum is outflow and the second is inflow. For example, if \(d_i<0\), then \(-d_i>0\), so the block must send out more TEU than it receives. If \(d_i>0\), its required net outflow is negative, which means it must receive more TEU than it sends.

The final load is its initial load minus outflow plus inflow. It must remain between zero and capacity:

$$
0\le L_i
-\sum_{\substack{j\in B\\j\ne i}}x_{ij}
+\sum_{\substack{j\in B\\j\ne i}}x_{ji}
\le U_i,
\qquad i\in B.
$$

The movement quantities must be nonnegative integers:

$$
x_{ij}\in\mathbb{Z}_{\ge 0},
\qquad i,j\in B,\ i\ne j.
$$

The flow-balance equations imply a necessary condition for feasibility. Summing them over all blocks counts every inter-block movement once as outflow and once as inflow, so all movement terms cancel. Therefore, \(\sum_{i\in B}d_i=0\) is necessary. In this example, \(\sum_i d_i=30\), so this condition fails. Also, the balance equation implies a final load of \(L_i+d_i\) for each block. Block 4 would have final load \(110+25=135\) TEU, exceeding its 120 TEU capacity. Either conflict is sufficient to make the model infeasible.

### Model size and code mapping

With \(n\) blocks, the model has \(n(n-1)\) directed movement variables and, as written, three block-level constraints per block: one flow-balance equality and two final-load inequalities. Thus the number of variables grows quadratically with the number of blocks, while the number of these constraints grows linearly. This describes model size, not a guaranteed solver runtime; runtime depends on the instance and the solver.

In the code, `movement_vars[(i, j)]` represents \(x_{ij}\), where `i` and `j` are zero-based matrix positions; `movement_costs[i, j]` represents \(c_{ij}\). Each `Block` stores \(L_i\), \(U_i\), and \(d_i\). The `outflow`, `inflow`, and `final_load` expressions implement the corresponding sums and load equation. A well-defined objective does not guarantee feasibility: the solver status determines whether a valid plan exists.

In [ ]:
def solve_network_flow_optimization(
    yard_state: YardState,
) -> Tuple[Dict[Tuple[int, int], int], Optional[float], str]:
    """Solve the integer minimum-cost network-flow model."""
    prob = pulp.LpProblem(
        "Yard_Housekeeping_Network_Flow",
        pulp.LpMinimize,
    )

    # Each variable represents a directed shipment; self-movements have no role.
    movement_vars = {}
    for i in range(yard_state.num_blocks):
        for j in range(yard_state.num_blocks):
            if i != j:
                movement_vars[(i, j)] = pulp.LpVariable(
                    f"move_{i + 1}_{j + 1}",
                    0,
                    None,
                    "Integer",
                )

    prob += pulp.lpSum(
        yard_state.movement_costs[i, j] * movement_vars[(i, j)]
        for i in range(yard_state.num_blocks)
        for j in range(yard_state.num_blocks)
        if i != j
    ), "Total_Movement_Cost"

    for i, block in enumerate(yard_state.blocks):
        outflow = pulp.lpSum(
            movement_vars[(i, j)]
            for j in range(yard_state.num_blocks)
            if i != j
        )
        inflow = pulp.lpSum(
            movement_vars[(j, i)]
            for j in range(yard_state.num_blocks)
            if i != j
        )

        # This equality enforces the exact signed net-flow requirement.
        prob += (
            outflow - inflow == -block.demand,
            f"Flow_Conservation_Block_{block.id}",
        )

        # The same net flow determines the final load, which must be physical.
        final_load = block.current_load - outflow + inflow
        prob += (
            final_load >= 0,
            f"Nonnegative_Final_Load_Block_{block.id}",
        )
        prob += (
            final_load <= block.capacity,
            f"Capacity_Block_{block.id}",
        )

    print("Solving the network-flow optimization model...")
    prob.solve(pulp.PULP_CBC_CMD(msg=False))
    solver_status = pulp.LpStatus[prob.status]

    if solver_status != "Optimal":
        return {}, None, solver_status

    solution = {}
    for (i, j), variable in movement_vars.items():
        amount = variable.varValue
        if amount is not None and amount > 0:
            solution[(i + 1, j + 1)] = int(round(amount))

    total_movement_cost = float(pulp.value(prob.objective))
    return solution, total_movement_cost, solver_status

In [ ]:
solution, total_cost, solver_status = solve_network_flow_optimization(yard)

print(f"\nSolver status: {solver_status}")
if solver_status == "Optimal":
    print(f"Total movement cost: {total_cost:.0f} cost units")
    print("\nOptimal container movements:")
    for (from_block, to_block), amount in sorted(solution.items()):
        unit_cost = yard.movement_costs[from_block - 1, to_block - 1]
        movement_cost = amount * unit_cost
        print(
            f"Move {amount} TEU: Block {from_block} to Block {to_block} "
            f"(cost: {movement_cost:.0f} cost units)"
        )
else:
    print(
        "No feasible movement plan or optimal cost is available for the "
        "stated demands and capacity limits."
    )

In [ ]:
def calculate_final_state(yard_state: YardState, movements: Dict) -> YardState:
    """Return a new yard state after applying the specified movements."""
    final_blocks = [
        Block(
            id=block.id,
            current_load=block.current_load,
            capacity=block.capacity,
            demand=block.demand,
        )
        for block in yard_state.blocks
    ]

    block_positions = {
        block.id: position for position, block in enumerate(final_blocks)
    }
    for (from_block, to_block), amount in movements.items():
        source_idx = block_positions[from_block]
        destination_idx = block_positions[to_block]
        final_blocks[source_idx].current_load -= amount
        final_blocks[destination_idx].current_load += amount

    return YardState(
        blocks=final_blocks,
        movement_costs=yard_state.movement_costs.copy(),
    )


if solver_status == "Optimal":
    final_yard = calculate_final_state(yard, solution)
    print("\nComparison of initial and final block loads:")
    print(
        "{:<8} {:<16} {:<16} {:<16} {:<16} {:<16}".format(
            "Block",
            "Initial (TEU)",
            "Final (TEU)",
            "Capacity (TEU)",
            "Initial use",
            "Final use",
        )
    )
    print("-" * 88)

    for initial, final in zip(yard.blocks, final_yard.blocks):
        print(
            "{:<8} {:<16d} {:<16d} {:<16d} {:<16.1%} {:<16.1%}".format(
                f"Block {initial.id}",
                initial.current_load,
                final.current_load,
                final.capacity,
                initial.utilization,
                final.utilization,
            )
        )
else:
    final_yard = None
    print(
        "\nFinal block loads are not reported because no feasible plan exists."
    )

## Interpreting the optimization result

A movement plan is meaningful only when the solver status is `Optimal`. In that case, the reported flows satisfy every exact net-flow equation and every final-load bound, and the objective is the sum of movement quantities multiplied by their unit costs. If the status is `Infeasible`, no plan satisfies all of those requirements simultaneously; reporting a movement cost or final state as a feasible result would be misleading.

The optimal plan need not be unique. If several feasible plans have the same minimum cost, the solver may return any one of them. The minimum objective value remains the same even when individual movements differ. Zero-cost movements can also permit multiple optimal movement descriptions, so interpretation should focus on feasibility and the objective as well as the particular plan returned.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
ax1, ax2, ax3, ax4 = axes.ravel()

block_labels = [f"Block {block.id}" for block in yard.blocks]
positions = np.arange(len(block_labels))
width = 0.36
initial_utilization = [block.utilization for block in yard.blocks]

ax1.bar(
    positions - width / 2,
    initial_utilization,
    width,
    label="Initial",
    alpha=0.85,
)
ax1.set_xticks(positions)
ax1.set_xticklabels(block_labels)
ax1.set_xlabel("Yard block")
ax1.set_ylabel("Capacity utilization")
ax1.set_title("Initial block utilization")
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
ax1.grid(axis="y", alpha=0.3)

if solver_status == "Optimal":
    final_utilization = [block.utilization for block in final_yard.blocks]
    ax1.bar(
        positions + width / 2,
        final_utilization,
        width,
        label="Final",
        alpha=0.85,
    )
    ax1.legend()
else:
    ax1.text(
        0.5,
        0.92,
        "Final utilization unavailable: no feasible plan",
        transform=ax1.transAxes,
        ha="center",
        va="top",
        wrap=True,
    )

if solver_status == "Optimal" and solution:
    movement_labels = [
        f"{source} to {destination}"
        for source, destination in solution
    ]
    movement_amounts = list(solution.values())
    ax2.barh(movement_labels, movement_amounts, color="tab:blue", alpha=0.8)
    ax2.set_xlabel("Moved quantity (TEU)")
    ax2.set_ylabel("Directed movement")
    ax2.set_title("Optimal container movements")
else:
    ax2.text(
        0.5,
        0.5,
        "No feasible movement plan to display",
        transform=ax2.transAxes,
        ha="center",
        va="center",
        wrap=True,
    )
    ax2.set_title("Container movements")
    ax2.set_xticks([])
    ax2.set_yticks([])

if solver_status == "Optimal" and solution:
    movement_costs = [
        amount * yard.movement_costs[source - 1, destination - 1]
        for (source, destination), amount in solution.items()
    ]
    movement_labels = [
        f"{source} to {destination}"
        for source, destination in solution
    ]
    ax3.pie(
        movement_costs,
        labels=movement_labels,
        autopct="%1.1f%%",
    )
    ax3.set_title(f"Movement-cost shares\nTotal: {total_cost:.0f} cost units")
else:
    ax3.text(
        0.5,
        0.5,
        "No feasible plan; movement costs are unavailable",
        transform=ax3.transAxes,
        ha="center",
        va="center",
        wrap=True,
    )
    ax3.set_title("Movement-cost breakdown")
    ax3.set_xticks([])
    ax3.set_yticks([])

demands = [block.demand for block in yard.blocks]
demand_colors = [
    "tab:blue" if demand > 0 else "tab:orange" if demand < 0 else "tab:gray"
    for demand in demands
]
bars = ax4.bar(block_labels, demands, color=demand_colors, alpha=0.8)
ax4.axhline(0, color="black", linewidth=0.8)
ax4.set_xlabel("Yard block")
ax4.set_ylabel("Signed demand (TEU)")
ax4.set_title("Net-flow requirements")
ax4.grid(axis="y", alpha=0.3)
ax4.legend(
    handles=[
        Patch(facecolor="tab:blue", label="Positive demand: net inflow"),
        Patch(facecolor="tab:orange", label="Negative demand: net outflow"),
        Patch(facecolor="tab:gray", label="Zero demand"),
    ],
    loc="best",
)
for bar, demand in zip(bars, demands):
    offset = 1 if demand >= 0 else -1
    ax4.text(
        bar.get_x() + bar.get_width() / 2,
        demand + offset,
        f"{demand:+d}",
        ha="center",
        va="bottom" if demand >= 0 else "top",
    )

fig.suptitle(
    "Yard housekeeping model diagnostics",
    fontsize=14,
)
fig.tight_layout()
plt.show()

## Sensitivity analysis

Sensitivity analysis examines whether a solution changes when model parameters change. Here, every movement cost is multiplied by the same positive factor. This preserves the relative ranking of all movement costs and leaves the flow-balance and capacity constraints unchanged. Therefore, a uniform positive cost multiplier cannot change feasibility. If a feasible instance has multiple optimal plans, the set of minimum-cost plans also remains the same, although the objective value is multiplied by that factor.

The experiment keeps the original block data fixed and resolves the same network-flow model for each multiplier. It records the solver status for every run and reports movement costs only when an optimal feasible solution exists. Since the example's infeasibility is caused by flow-balance and capacity requirements, changing only the objective coefficients cannot resolve it.

In [ ]:
def sensitivity_analysis(yard_state: YardState) -> Dict:
    """Resolve the same model under uniform movement-cost multipliers."""
    cost_multipliers = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
    results = {}

    for multiplier in cost_multipliers:
        scaled_yard = YardState(
            blocks=yard_state.blocks,
            movement_costs=yard_state.movement_costs * multiplier,
        )
        solution, total_cost, solver_status = (
            solve_network_flow_optimization(scaled_yard)
        )
        results[multiplier] = {
            "status": solver_status,
            "total_cost": total_cost,
            "num_moves": len(solution),
            "solution": solution,
        }

    return results


sensitivity_results = sensitivity_analysis(yard)

print("\nSensitivity analysis: uniform movement-cost multipliers")
print(
    "{:<12} {:<16} {:<14} {:<16}".format(
        "Multiplier",
        "Solver status",
        "Move arcs",
        "Total cost",
    )
)
print("-" * 62)

for multiplier, result in sensitivity_results.items():
    displayed_cost = (
        f"{result['total_cost']:.2f}"
        if result["total_cost"] is not None
        else "Unavailable"
    )
    print(
        "{:<12.2f} {:<16} {:<14d} {:<16}".format(
            multiplier,
            result["status"],
            result["num_moves"],
            displayed_cost,
        )
    )

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

multipliers = list(sensitivity_results)
feasibility = [
    1 if result["status"] == "Optimal" else 0
    for result in sensitivity_results.values()
]

ax1.plot(
    multipliers,
    feasibility,
    marker="o",
    linewidth=2,
    color="tab:blue",
)
ax1.set_xticks(multipliers)
ax1.set_xlabel("Uniform movement-cost multiplier")
ax1.set_ylabel("Feasible optimum (1 = yes, 0 = no)")
ax1.set_yticks([0, 1])
ax1.set_ylim(-0.15, 1.15)
ax1.set_title("Feasibility across cost scenarios")
ax1.grid(True, alpha=0.3)

all_moves = sorted(
    {
        move
        for result in sensitivity_results.values()
        for move in result["solution"]
    }
)

if all_moves:
    move_matrix = np.zeros((len(all_moves), len(multipliers)))
    for row, move in enumerate(all_moves):
        for column, multiplier in enumerate(multipliers):
            move_matrix[row, column] = sensitivity_results[multiplier][
                "solution"
            ].get(move, 0)

    image = ax2.imshow(move_matrix, cmap="YlOrRd", aspect="auto")
    ax2.set_xticks(range(len(multipliers)))
    ax2.set_xticklabels([f"{multiplier:g}x" for multiplier in multipliers])
    ax2.set_yticks(range(len(all_moves)))
    ax2.set_yticklabels(
        [f"{source} to {destination}" for source, destination in all_moves]
    )
    ax2.set_xlabel("Uniform movement-cost multiplier")
    ax2.set_ylabel("Directed movement")
    ax2.set_title("Selected movement quantities")
    colorbar = fig.colorbar(image, ax=ax2)
    colorbar.set_label("Moved quantity (TEU)")
else:
    ax2.text(
        0.5,
        0.5,
        "No feasible movement plans are available to compare",
        transform=ax2.transAxes,
        ha="center",
        va="center",
        wrap=True,
    )
    ax2.set_title("Selected movement quantities")
    ax2.set_xticks([])
    ax2.set_yticks([])

fig.tight_layout()
plt.show()

## Feasibility verification

Verification independently checks the reported plan against the model's movement-variable domain, flow equations, final-load bounds, and objective calculation. These checks connect the solver's plan back to the original data. They verify feasibility and objective consistency; global optimality is established by the solver's optimization result, not by these independent arithmetic checks. If no optimal feasible plan exists, there is no plan to verify.

In [ ]:
def verify_optimality(
    yard_state: YardState,
    solution: Dict[Tuple[int, int], int],
    total_cost: Optional[float],
    solver_status: str,
) -> bool:
    """Check the plan's variable domain, flow, capacity, and cost conditions."""
    print("\n=== FEASIBILITY AND COST VERIFICATION ===")

    if solver_status != "Optimal" or total_cost is None:
        print(f"Solver status: {solver_status}")
        print("No feasible optimal plan is available to verify.")
        return False

    valid_movements = True
    allowed_ids = {block.id for block in yard_state.blocks}
    for (source, destination), amount in solution.items():
        valid_arc = (
            source in allowed_ids
            and destination in allowed_ids
            and source != destination
        )
        valid_quantity = (
            isinstance(amount, (int, np.integer))
            and amount >= 0
        )
        valid_movements = valid_movements and valid_arc and valid_quantity

    print("\n1. Movement-variable domain check:")
    print(
        "   "
        + ("PASS" if valid_movements else "FAIL")
        + ": movements use distinct valid blocks and nonnegative integer TEU"
    )
    if not valid_movements:
        print("Verification result: one or more checks fail.")
        return False

    demand_satisfied = True
    print("\n2. Net-flow requirement check:")
    for block in yard_state.blocks:
        outflow = sum(
            amount
            for (source, _), amount in solution.items()
            if source == block.id
        )
        inflow = sum(
            amount
            for (_, destination), amount in solution.items()
            if destination == block.id
        )
        net_outflow = outflow - inflow
        expected_net_outflow = -block.demand
        satisfied = np.isclose(
            net_outflow,
            expected_net_outflow,
            rtol=0.0,
            atol=1e-9,
        )
        demand_satisfied = demand_satisfied and satisfied
        label = "PASS" if satisfied else "FAIL"
        print(
            f"   {label}: Block {block.id}, net outflow {net_outflow:.1f} TEU; "
            f"required {expected_net_outflow:.1f} TEU"
        )

    final_yard = calculate_final_state(yard_state, solution)
    capacity_satisfied = True
    print("\n3. Final-load capacity check:")
    for block in final_yard.blocks:
        satisfied = 0 <= block.current_load <= block.capacity
        capacity_satisfied = capacity_satisfied and satisfied
        label = "PASS" if satisfied else "FAIL"
        print(
            f"   {label}: Block {block.id}, final load {block.current_load} TEU; "
            f"capacity {block.capacity} TEU"
        )

    calculated_cost = sum(
        amount * yard_state.movement_costs[source - 1, destination - 1]
        for (source, destination), amount in solution.items()
    )
    cost_correct = np.isclose(
        calculated_cost,
        total_cost,
        rtol=1e-9,
        atol=1e-9,
    )
    print("\n4. Objective calculation check:")
    print(f"   Recalculated cost: {calculated_cost:.6g} cost units")
    print(f"   Solver objective:  {total_cost:.6g} cost units")
    print(f"   {'PASS' if cost_correct else 'FAIL'}: objective values agree")

    verified = demand_satisfied and capacity_satisfied and cost_correct
    print(
        "\nFeasibility and cost verification: "
        + ("all checks pass." if verified else "one or more checks fail.")
    )
    return verified


is_optimal = verify_optimality(
    yard,
    solution,
    total_cost,
    solver_status,
)

## Summary and key insights

The notebook formulates yard housekeeping as an integer minimum-cost network-flow problem. Directed movement variables represent TEU transfers; flow-balance equations enforce each block's exact signed demand; final-load bounds require nonnegative loads no greater than block capacities; and the objective sums movement quantities multiplied by their per-TEU costs.

For the supplied example, the signed demands sum to 30 TEU, so their required net outflows sum to −30 TEU rather than zero. Inter-block movements alone cannot satisfy that imbalance. Independently, Block 4's required net inflow would result in a final load of 135 TEU against a capacity of 120 TEU. The instance therefore has no feasible movement plan under this model. Uniform cost multipliers cannot resolve these conflicts because they change objective coefficients, not the feasible region.

The model is deterministic and covers a single planning period. It does not represent holding costs, time-expanded decisions, uncertain demand, equipment availability, handling limits, or movement sequencing. Representing additional operational requirements would require extending the data and constraints while retaining the network-flow modeling approach.